In [1]:
# 04_robustness.ipynb
# Robustness for both studies:
#   (1) Bucket-boundary sensitivity of the base grade (Study 1): counts 5/7/10 and fixed-PD cuts.
#   (2) Regularized proportional-odds (hand-rolled, L2) so the full ordinal model converges (Study 2),
#       with H5 band asymmetry cross-checked across three estimators.
#   (3) Indicator-choice swap: leverage+profitability vs liquidity+activity, in both studies.
# All code in a single cell; grayscale figures saved as png + pdf at dpi 600.

import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.io import arff
from scipy import stats
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, cohen_kappa_score
import statsmodels.api as sm
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns

DATA = "../data"; FIG = "../results/figures"; TAB = "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="white", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8, "axes.grid": False})
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}/{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight"); plt.close(fig)

# =================================================================
# Study 1 data
# =================================================================
def load_year(n):
    data, _ = arff.loadarff(f"{DATA}/polish/{n}year.arff"); df = pd.DataFrame(data)
    cls = [c for c in df.columns if c.lower() == "class"][0]
    df[cls] = df[cls].apply(lambda b: int(b.decode()) if isinstance(b, bytes) else int(b))
    fe = [c for c in df.columns if c != cls]
    df.columns = [f"Attr{i+1}" for i in range(len(fe))] + ["target"]; df["horizon"] = n
    return df
pooled = pd.concat([load_year(n) for n in range(1, 6)], ignore_index=True)
feats = [c for c in pooled.columns if c.startswith("Attr")]
y1 = pooled["target"].astype(int).values
X1 = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(pooled[feats]), columns=feats)
Hd = pd.get_dummies(pooled["horizon"], prefix="h", drop_first=True).astype(float).reset_index(drop=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def delong(y_true, p1, p2):
    y_true = np.asarray(y_true); pos = y_true == 1; neg = ~pos; m = pos.sum(); n = neg.sum()
    def st(p):
        xp, xn = p[pos], p[neg]
        V10 = np.array([(np.sum(xn < xi) + 0.5*np.sum(xn == xi))/n for xi in xp])
        V01 = np.array([(np.sum(xp > xj) + 0.5*np.sum(xp == xj))/m for xj in xn])
        return V10.mean(), V10, V01
    a1, v10a, v01a = st(p1); a2, v10b, v01b = st(p2)
    S10 = np.cov(np.vstack([v10a, v10b])); S01 = np.cov(np.vstack([v01a, v01b]))
    var = S10/m + S01/n; v = var[0, 0] + var[1, 1] - 2*var[0, 1]
    z = (a1 - a2)/np.sqrt(v) if v > 0 else 0.0
    return float(2*(1 - stats.norm.cdf(abs(z))))

def oof(proxies):
    bf = [c for c in feats if c not in proxies]
    bp = cross_val_predict(LogisticRegression(max_iter=3000, class_weight="balanced"),
         StandardScaler().fit_transform(X1[bf]), y1, cv=cv, method="predict_proba")[:, 1]
    S = cross_val_predict(LogisticRegression(max_iter=3000, class_weight="balanced"),
         StandardScaler().fit_transform(X1[proxies]), y1, cv=cv, method="predict_proba")[:, 1]
    return bp, S

def grade_eval(G, S):
    D = pd.DataFrame({"y": y1, "G": G.astype(float), "S": S}).reset_index(drop=True)
    p0 = cross_val_predict(LogisticRegression(max_iter=3000), pd.concat([D[["G"]], Hd], axis=1).values, y1, cv=cv, method="predict_proba")[:, 1]
    p1 = cross_val_predict(LogisticRegression(max_iter=3000), pd.concat([D[["G", "S"]], Hd], axis=1).values, y1, cv=cv, method="predict_proba")[:, 1]
    m0 = sm.Logit(D["y"], sm.add_constant(pd.concat([D[["G"]], Hd], axis=1).astype(float))).fit(disp=0)
    m1 = sm.Logit(D["y"], sm.add_constant(pd.concat([D[["G", "S"]], Hd], axis=1).astype(float))).fit(disp=0)
    LR = 2*(m1.llf - m0.llf)
    D["mid"] = np.isin(G, [2, 3, 4]).astype(float); D["S_mid"] = D["S"]*D["mid"]
    m4 = sm.Logit(D["y"], sm.add_constant(pd.concat([D[["G", "S", "mid", "S_mid"]], Hd], axis=1).astype(float))).fit(disp=0)
    return {"AUC_M0": roc_auc_score(y1, p0), "AUC_M1": roc_auc_score(y1, p1),
            "DeLong_p": delong(y1, p1, p0), "LR": float(LR), "H4_z": float(m4.tvalues["S_mid"])}

# ---- (1) bucket sensitivity (primary proxies Attr1,Attr2) ----
bp, S = oof(["Attr1", "Attr2"])
rows = []
for k in [5, 7, 10]:
    G = pd.qcut(pd.Series(bp), k, labels=False, duplicates="drop").values
    r = grade_eval(G, S); r = {"scheme": f"quantile {k}", **r}; rows.append(r)
cuts = [0, 0.01, 0.02, 0.05, 0.10, 0.20, 1.01]
Gf = pd.cut(pd.Series(bp), bins=cuts, labels=False, include_lowest=True)
Gf = Gf.fillna(Gf.max()).astype(int).values
rows.append({"scheme": "fixed PD cuts", **grade_eval(Gf, S)})
bucket = pd.DataFrame(rows)
for c in ["AUC_M0", "AUC_M1", "LR", "H4_z"]: bucket[c] = bucket[c].round(3)
bucket["DeLong_p"] = bucket["DeLong_p"].map(lambda v: f"{v:.2e}")
bucket.to_csv(f"{TAB}/robust_bucket_sensitivity.csv", index=False)
print("Bucket sensitivity:\n", bucket.to_string(index=False))

# ---- Fig 10: H4 interaction z across bucket schemes ----
fig, ax = plt.subplots(figsize=(4.6, 3.0))
zz = [float(v) for v in bucket["H4_z"]]
ax.bar(bucket["scheme"], zz, color=["0.3", "0.3", "0.3", "0.7"], edgecolor="black", linewidth=0.7)
ax.axhline(1.96, color="0.5", linestyle="--", linewidth=0.8); ax.axhline(-1.96, color="0.5", linestyle="--", linewidth=0.8)
ax.set_ylabel("H4 interaction z  (S x mid-band)"); ax.tick_params(axis="x", labelrotation=20)
save_fig(fig, "fig10_robust_bucket_h4z")

# =================================================================
# Study 2 data + regularized proportional-odds
# =================================================================
df = pd.read_csv(f"{DATA}/corporate_rating.csv"); df.columns = [c.strip().replace(" ", "_") for c in df.columns]
meta = ["Rating", "Name", "Symbol", "Rating_Agency_Name", "Date", "Sector"]
f2 = [c for c in df.columns if c not in meta]
order = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]; K = len(order)
df["R"] = df["Rating"].replace({"CC": "CCC", "C": "CCC", "D": "CCC"})
df["y"] = df["R"].map({r: i for i, r in enumerate(order)})
df["band"] = df["R"].apply(lambda r: "mid" if r in ("BBB", "BB", "B") else "extreme")
y2 = df["y"].values
Xz = pd.DataFrame(StandardScaler().fit_transform(df[f2].fillna(df[f2].median())), columns=f2, index=df.index).clip(-5, 5)
agency = pd.get_dummies(df["Rating_Agency_Name"], prefix="ag", drop_first=True).astype(float)
sector = pd.get_dummies(df["Sector"], prefix="sec", drop_first=True).astype(float)

def fit_po(X, yy, lam=2.0):
    X = np.asarray(X, float); n, p = X.shape
    def unpack(th):
        a0 = th[0]; dl = th[1:K-1]; beta = th[K-1:]
        return np.concatenate([[a0], a0 + np.cumsum(np.exp(dl))]), beta
    def nll(th):
        cuts, beta = unpack(th); eta = X @ beta
        F = expit(cuts[None, :] - eta[:, None]); P = np.empty((n, K))
        P[:, 0] = F[:, 0]; P[:, 1:K-1] = F[:, 1:] - F[:, :-1]; P[:, K-1] = 1 - F[:, -1]
        P = np.clip(P, 1e-12, 1)
        return -np.log(P[np.arange(n), yy]).sum() + lam*np.sum(beta**2)
    r = minimize(nll, np.concatenate([[0.0], np.zeros(K-2), np.zeros(p)]), method="L-BFGS-B", options={"maxiter": 500})
    return unpack(r.x)
def predict_po(cuts, beta, X):
    X = np.asarray(X, float); eta = X @ beta; F = expit(cuts[None, :] - eta[:, None]); n = len(eta); P = np.empty((n, K))
    P[:, 0] = F[:, 0]; P[:, 1:K-1] = F[:, 1:] - F[:, :-1]; P[:, K-1] = 1 - F[:, -1]
    return P.argmax(1)
def cv_po(X):
    X = X.reset_index(drop=True); pred = np.empty(len(y2), int)
    for tr, te in cv.split(X, y2):
        cuts, beta = fit_po(X.iloc[tr].values, y2[tr]); pred[te] = predict_po(cuts, beta, X.iloc[te].values)
    return pred

XfullFE = pd.concat([Xz[f2], agency, sector], axis=1)
pred_po = cv_po(XfullFE)
reg = pd.DataFrame([{"model": "Regularized ordinal full+FE",
                     "QWK": round(cohen_kappa_score(y2, pred_po, weights="quadratic"), 4),
                     "within1": round(float(np.mean(np.abs(y2 - pred_po) <= 1)), 4)}])
reg.to_csv(f"{TAB}/robust_regularized_ordinal.csv", index=False)
print("\nRegularized ordinal (converges):\n", reg.to_string(index=False))

# ---- H5 across three estimators (band-wise within-1 loss, 2 proxies vs full) ----
from statsmodels.miscmodels.ordinal_model import OrderedModel
proxies = ["debtRatio", "returnOnAssets"]
def rf_oof(cols): return cross_val_predict(RandomForestClassifier(300, random_state=0, n_jobs=-1), Xz[cols].values, y2, cv=cv)
def ol_oof(Xdf):                                  # statsmodels ordered logit, OOF argmax
    Xdf = Xdf.reset_index(drop=True); pred = np.empty(len(y2), int)
    for tr, te in cv.split(Xdf, y2):
        m = OrderedModel(y2[tr], Xdf.iloc[tr], distr="logit").fit(method="lbfgs", disp=0, maxiter=300)
        pred[te] = np.asarray(m.predict(Xdf.iloc[te])).argmax(1)
    return pred
po_full, po_two = pred_po, cv_po(Xz[proxies])
methods = {"Random forest": (rf_oof(f2), rf_oof(proxies)),
           "Ordered logit": (ol_oof(XfullFE), ol_oof(Xz[proxies])),
           "Regularized ordinal (PO)": (po_full, po_two)}
h5_rows = []
for name, (pf, pt) in methods.items():
    for b in ["mid", "extreme"]:
        m = df["band"].values == b
        loss = 100*(np.mean(np.abs(y2[m]-pf[m]) <= 1) - np.mean(np.abs(y2[m]-pt[m]) <= 1))
        h5_rows.append({"method": name, "band": b, "loss_pp": round(float(loss), 1)})
h5 = pd.DataFrame(h5_rows); h5.to_csv(f"{TAB}/robust_h5_three_methods.csv", index=False)
print("\nH5 band loss across three estimators:\n", h5.to_string(index=False))

# ---- Fig 11: H5 band loss across three methods ----
piv = h5.pivot(index="method", columns="band", values="loss_pp").reindex(["Random forest", "Ordered logit", "Regularized ordinal (PO)"])
fig, ax = plt.subplots(figsize=(5.2, 3.0)); x = np.arange(len(piv)); w = 0.38
ax.bar(x - w/2, piv["mid"], w, color="0.3", edgecolor="black", linewidth=0.6, label="Mid band")
ax.bar(x + w/2, piv["extreme"], w, color="0.7", edgecolor="black", linewidth=0.6, label="Extremes")
ax.set_xticks(x); ax.set_xticklabels(piv.index, rotation=12, ha="right", fontsize=8)
ax.set_ylabel("2-proxy info loss (pp, within-1)"); ax.legend(fontsize=8, frameon=False)
save_fig(fig, "fig11_robust_h5_three_methods")

# =================================================================
# (3) Indicator-choice swap: leverage+profitability vs liquidity+activity
# =================================================================
# Study 1 incremental LR for each pair
bp_lp, S_lp = bp, S                           # Attr1 (ROA), Attr2 (leverage) already computed
G_lp = pd.qcut(pd.Series(bp_lp), 7, labels=False, duplicates="drop").values
LR_lp = grade_eval(G_lp, S_lp)["LR"]
bp_la, S_la = oof(["Attr4", "Attr9"])          # Attr4 current ratio (liquidity), Attr9 asset turnover (activity)
G_la = pd.qcut(pd.Series(bp_la), 7, labels=False, duplicates="drop").values
LR_la = grade_eval(G_la, S_la)["LR"]
# Study 2 recovery QWK for each pair (RF)
qwk_lp = cohen_kappa_score(y2, rf_oof(["debtRatio", "returnOnAssets"]), weights="quadratic")
qwk_la = cohen_kappa_score(y2, rf_oof(["currentRatio", "assetTurnover"]), weights="quadratic")
swap = pd.DataFrame([
    {"indicator_pair": "Leverage + Profitability", "study1_incremental_LR": round(LR_lp, 1), "study2_recovery_QWK": round(qwk_lp, 4)},
    {"indicator_pair": "Liquidity + Activity",     "study1_incremental_LR": round(LR_la, 1), "study2_recovery_QWK": round(qwk_la, 4)},
])
swap.to_csv(f"{TAB}/robust_indicator_choice.csv", index=False)
print("\nIndicator-choice comparison:\n", swap.to_string(index=False))

# ---- Fig 12: indicator-choice comparison (two panels) ----
fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.0))
axes[0].bar(["Lev+Prof", "Liq+Act"], swap["study1_incremental_LR"], color=["0.25", "0.7"], edgecolor="black", linewidth=0.7)
axes[0].set_ylabel("Study 1  incremental LR statistic")
axes[1].bar(["Lev+Prof", "Liq+Act"], swap["study2_recovery_QWK"], color=["0.25", "0.7"], edgecolor="black", linewidth=0.7)
axes[1].set_ylabel("Study 2  recovery QWK")
save_fig(fig, "fig12_robust_indicator_choice")

print("\n[04_robustness] done. Figures + tables written to ../results/.")


Bucket sensitivity:
        scheme  AUC_M0  AUC_M1 DeLong_p      LR   H4_z
   quantile 5   0.730   0.736 6.28e-08 104.152  0.377
   quantile 7   0.734   0.738 7.56e-06  78.320  5.065
  quantile 10   0.735   0.737 7.85e-04  68.029  2.278
fixed PD cuts   0.563   0.690 0.00e+00 733.909 -2.986



Regularized ordinal (converges):
                       model   QWK  within1
Regularized ordinal full+FE 0.399   0.8438


/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1189: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = fit_method(


/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1189: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = fit_method(
/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1189: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = fit_method(


/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1189: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = fit_method(


/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1189: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = fit_method(
/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1189: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = fit_method(


/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1189: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = fit_method(
/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1189: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = fit_method(



H5 band loss across three estimators:
                   method    band  loss_pp
           Random forest     mid     11.6
           Random forest extreme     18.4
           Ordered logit     mid      4.1
           Ordered logit extreme      8.7
Regularized ordinal (PO)     mid      4.2
Regularized ordinal (PO) extreme      8.3



Indicator-choice comparison:
           indicator_pair  study1_incremental_LR  study2_recovery_QWK
Leverage + Profitability                   78.3               0.3479
    Liquidity + Activity                    0.7               0.1445



[04_robustness] done. Figures + tables written to ../results/.
